## Finetuned Single Test

In [ ]:
import torch
import torchaudio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_path = "whisper-finetuned-manual"

processor = WhisperProcessor.from_pretrained(model_path)
model = WhisperForConditionalGeneration.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(128, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1280)
      (layers): ModuleList(
        (0-31): 32 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (fc2): Linear(in_features=5120, out_features=1280, bias=Tr

In [ ]:
audio_path = "segments/aud (164)_seg1.wav"

waveform, sr = torchaudio.load(audio_path)
waveform = waveform.mean(dim=0, keepdim=True)
waveform = torchaudio.functional.resample(waveform, sr, 16000)
waveform = waveform.squeeze().numpy()

inputs = processor.feature_extractor(waveform, sampling_rate=16000, return_tensors="pt")
input_features = inputs.input_features.to(device)

forced_decoder_ids = processor.get_decoder_prompt_ids(language="russian", task="transcribe")

with torch.no_grad():
    predicted_ids = model.generate(
        input_features,
        forced_decoder_ids=forced_decoder_ids,
        max_length=448
    )


transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print("Распознано:")
print(transcription)


## WER Comparison

In [ ]:
import json

def load_manifest_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

eval_data = load_manifest_jsonl("manifest.jsonl")[:100]  

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import whisper
import torch

def transcribe_whisper(model, processor, audio_path):
    audio = whisper.load_audio(audio_path)
    audio = whisper.pad_or_trim(audio)
    input_features = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
    input_features = torch.tensor(input_features).unsqueeze(0).to("cuda")


    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=processor.get_decoder_prompt_ids(language="russian", task="transcribe"),
            max_length=448
        )
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()

In [3]:
from evaluate import load
wer_metric = load("wer")

def compute_wer(model, processor, eval_data):
    predictions = []
    references = []
    for item in eval_data:
        text = transcribe_whisper(model, processor, item["audio_filepath"])
        predictions.append(text)
        references.append(item["text"])
    return wer_metric.compute(predictions=predictions, references=references)


In [ ]:
# Базовая модель
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3").to("cuda")
base_processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")

# Дообученная модель
finetuned_model = WhisperForConditionalGeneration.from_pretrained("whisper-finetuned-manual").to("cuda")
finetuned_processor = WhisperProcessor.from_pretrained("whisper-finetuned-manual")

base_wer = compute_wer(base_model, base_processor, eval_data)
finetuned_wer = compute_wer(finetuned_model, finetuned_processor, eval_data)

print(f"Base WER:      {base_wer:.4f}")
print(f"Finetuned WER: {finetuned_wer:.4f}")
